In [39]:
import numpy as np
from keras.datasets import mnist
from PIL import Image

In [47]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# nrmalize pixel values to binary
x_train = x_train.astype(np.float32)/255.0
x_test = x_test.astype(np.float32)/255.0

# One-hot encoding
def one_hot_encode(y,num_classes=10):
    return np.eye(num_classes)[y]

y_train_oh = one_hot_encode(y_train)
y_test_oh = one_hot_encode(y_test)

print("Train images:", x_train.shape)
print("Train labels:",y_train_oh.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test_oh.shape)

Train images: (60000, 28, 28)
Train labels: (60000, 10)
Test images: (10000, 28, 28)
Test labels: (10000, 10)


## Activation Functions

In [49]:
def sigmoid(x):
    return 1/(1 + np.exp(-x))
    
def sigmoid_derivative(sigmoid_output):
    return sigmoid_output*(1 - sigmoid_output)

def softmax(x):
    exps = np.exp(x - np.max(x))
    return exps/np.sum(exps)

def cross_entropy(predicted, target):
    return -np.sum(target*np.log(predicted + 1e-9))



## Convolution

In [50]:
# convolution operation
def conv2d(image, kernel, stride=1):
    kH, kW = kernel.shape
    iH, iW = image.shape
    oH = (iH - kH)//stride + 1
    oW = (iW - kW)//stride + 1
    output = np.zeros((oH, oW))
    for y in range(0, oH):
        for x in range(0, oW):
            region = image[y*stride:y*stride+kH, x*stride:x*stride+kW]
            output[y, x] = np.sum(region * kernel)
    return output

# Convolution layer with multiple filters
def conv_layer(image, filters, stride=1):
    output_maps = []
    for filt in filters:
        convolved = conv2d(image, filt, stride)
        activated = sigmoid(convolved)
        output_maps.append(activated)
    return np.array(output_maps)

# Initialize 2 filters with random values
np.random.seed(0)
filters = [np.random.randn(3, 3)*0.1 for _ in range(2)]

# Sample for test
sample_img = x_train[0]
conv_output = conv_layer(sample_img, filters)
print("Shape after conv layer:",conv_output.shape)


Shape after conv layer: (2, 26, 26)


## Average Pooling Layer with pool size 2x2 and stride 2

In [51]:
def average_pooling(feature_maps, size=2, stride=2):
    n_filters, h, w = feature_maps.shape
    pooled_h = (h - size)//stride + 1
    pooled_w = (w - size)//stride + 1
    pooled = np.zeros((n_filters, pooled_h, pooled_w))
    
    for f in range(n_filters):
        for y in range(pooled_h):
            for x in range(pooled_w):
                region = feature_maps[f, y*stride:y*stride+size, x*stride:x*stride+size]
                pooled[f, y, x] =np.mean(region)
                
    return pooled

# Sample for test
pooled_output = average_pooling(conv_output)
print("Shape after avg pooling:", pooled_output.shape)  # Should be (2, 13, 13)


Shape after avg pooling: (2, 13, 13)


## Flatten and 1x1 Convolution Layer with 10 output channels

In [52]:
def flatten(feature_maps):
    return feature_maps.flatten()

# 1x1 Convolution Layer
def fully_connected(flattened, weights, bias):
    return np.dot(weights, flattened)+bias

# Initialize weights and bias
np.random.seed(1)
fc_weights = np.random.randn(10, 338) * 0.1
fc_bias = np.zeros(10)

# sample
flat = flatten(pooled_output)
fc_out = fully_connected(flat, fc_weights, fc_bias)
probs = softmax(fc_out) 
print("Output probabilities:", probs)
print("Sum of probs:", np.sum(probs))
print("Predicted class:", np.argmax(probs))


Output probabilities: [0.21357469 0.07665425 0.06464049 0.20362415 0.07089453 0.0442681
 0.03571858 0.06327013 0.0688406  0.15851448]
Sum of probs: 1.0
Predicted class: 0


## Forward and Back

In [53]:
def forward(image):
    conv_out = conv_layer(image, filters)
    pool_out = average_pooling(conv_out)
    flat = flatten(pool_out)
    fc_out = fully_connected(flat, fc_weights, fc_bias)
    probs = softmax(fc_out)
    return probs, conv_out, pool_out, flat

def backprop_flatten(grad_flattened, original_shape=(2, 13, 13)):
    return grad_flattened.reshape(original_shape)

def backprop_avg_pooling(grad_pooled, original_input_shape=(2, 26, 26), pool_size=2, stride=2):
    n_filters, h, w = original_input_shape
    grad = np.zeros(original_input_shape)
    
    for f in range(n_filters):
        for y in range(grad_pooled.shape[1]):
            for x in range(grad_pooled.shape[2]):
                grad_value = grad_pooled[f, y, x]/(pool_size * pool_size)
                grad[f,
                     y*stride:y*stride+pool_size,
                     x*stride:x*stride+pool_size] +=grad_value
                
    return grad
    
def backprop_conv_layer(image, grad_conv_output, filters, stride=1):
    grad_filters = [np.zeros_like(f) for f in filters]
    for idx, filt in enumerate(filters):
        for y in range(grad_conv_output.shape[1]):
            for x in range(grad_conv_output.shape[2]):
                region = image[y:y+3, x:x+3]
                grad_filters[idx] +=grad_conv_output[idx, y, x]*region

    return grad_filters


## Train

In [54]:
def train_one_sample_full(image, label_oh, lr=0.01):
    global fc_weights, fc_bias, filters

    # Forward
    conv_out = conv_layer(image, filters)
    pool_out = average_pooling(conv_out)
    flat = flatten(pool_out)
    fc_out = fully_connected(flat, fc_weights, fc_bias)
    probs = softmax(fc_out)
    loss = cross_entropy(probs, label_oh)

    # Backprop of all layer
    
    # Output layer grad
    grad_output = probs - label_oh
    grad_fc_weights = np.outer(grad_output, flat)
    grad_fc_bias =grad_output

    # Flatten layer
    grad_flat = fc_weights.T @ grad_output
    grad_pool_out =backprop_flatten(grad_flat)

    # Avg Pooling
    grad_conv_out = backprop_avg_pooling(grad_pool_out, conv_out.shape)

    # Sigmoid activation
    conv_out_sigmoid =conv_out
    grad_sigmoid = sigmoid_derivative(conv_out_sigmoid)
    grad_conv_out*=grad_sigmoid

    # Convolution filter update
    grads_filters =backprop_conv_layer(image, grad_conv_out, filters)

    # Update weights
    fc_weights -= lr*grad_fc_weights
    fc_bias -= lr*grad_fc_bias
    for i in range(len(filters)):
        filters[i] -= lr*grads_filters[i]

    return loss, np.argmax(probs)


In [56]:
losses = []
correct = 0
for i in range(60000):
    img = x_train[i]
    label = y_train_oh[i]
    true_label = y_train[i]

    loss, pred = train_one_sample_full(img, label)
    losses.append(loss)
    if pred == true_label:
        correct += 1

print(f"Accuracy: {(correct/60000)*100}%")
print(f"Avg Loss: {np.mean(losses):}")


Accuracy: 85.83666666666666%
Avg Loss: 0.48759763554353913


In [57]:
def evaluate_model(x_data, y_data_oh, y_data):
    total_loss = 0
    correct = 0
    for i in range(len(x_data)):
        image = x_data[i]
        label_oh = y_data_oh[i]
        true_label = y_data[i]

        probs, _, _, _ = forward(image)
        loss = cross_entropy(probs, label_oh)
        pred = np.argmax(probs)

        total_loss += loss
        if pred == true_label:
            correct += 1

    avg_loss = total_loss / len(x_data)
    accuracy = correct / len(x_data)
    print(f"\nTest Accuracy: {accuracy*100}%")
    print(f"Test Avg Loss: {avg_loss}")


In [58]:
evaluate_model(x_test, y_test_oh, y_test)


Test Accuracy: 88.21%
Test Avg Loss: 0.3662125948586467


So, from the model, after training with 60000 images and testing with 10000 images, we got a test accuracy of 88.21% with a test average loss of 0.367